In [0]:
from pyspark.sql import functions as F
from datetime import datetime

BASE = "/Volumes/workspace/raw/landing"
lote = datetime.now().strftime("%Y%m%d_%H%M%S")
n = 50000

categorias = F.array(*[F.lit(c) for c in ["mercado", "transporte", "lazer", "saude", "contas", "educacao"]])
canais = F.array(*[F.lit(c) for c in ["app", "cartao", "pix", "boleto"]])

transacoes = (
    spark.range(n)
    .withColumn("id_transacao", F.concat(F.lit(lote), F.lit("-"), F.col("id").cast("string")))
    .withColumn("id_cliente", (F.rand(1) * 1000).cast("int") + 1)
    .withColumn("valor", F.round(F.rand(2) * 2000 + 1, 2))
    .withColumn("categoria", F.element_at(categorias, (F.rand(3) * 6).cast("int") + 1))
    .withColumn("canal", F.element_at(canais, (F.rand(4) * 4).cast("int") + 1))
    .withColumn("data_transacao",
        F.expr("current_timestamp() - make_interval(0, 0, 0, 0, 0, cast(rand(5) * 43200 as int), 0)"))
    .drop("id")
)

# Dados problemáticos de propósito
negativos = (transacoes.sample(0.02, seed=10)
             .withColumn("id_transacao", F.concat(F.col("id_transacao"), F.lit("-neg")))
             .withColumn("valor", -F.col("valor")))
sem_cliente = (transacoes.sample(0.01, seed=11)
               .withColumn("id_transacao", F.concat(F.col("id_transacao"), F.lit("-nul")))
               .withColumn("id_cliente", F.lit(None).cast("int")))
duplicadas = transacoes.sample(0.01, seed=12)

lote_final = (transacoes.unionByName(negativos)
              .unionByName(sem_cliente)
              .unionByName(duplicadas))

lote_final.write.mode("overwrite").json(f"{BASE}/transacoes/{lote}")
print(f"Lote {lote}: {lote_final.count()} registros")

In [0]:
%sql
SELECT
  count(*)                                AS total,
  count_if(valor < 0)                     AS negativos,
  count_if(id_cliente IS NULL)            AS sem_cliente,
  count(*) - count(DISTINCT id_transacao) AS duplicadas
FROM json.`/Volumes/workspace/raw/landing/transacoes/*/`

In [0]:
segmentos = F.array(F.lit("varejo"), F.lit("alta_renda"), F.lit("pj"))
cidades = F.array(*[F.lit(c) for c in ["Belo Horizonte", "São Paulo", "Rio de Janeiro", "Curitiba", "Salvador"]])

clientes = (
    spark.range(1, 1001).withColumnRenamed("id", "id_cliente")
    .withColumn("id_cliente", F.col("id_cliente").cast("int"))
    .withColumn("nome", F.concat(F.lit("Cliente "), F.col("id_cliente").cast("string")))
    .withColumn("segmento", F.element_at(segmentos, (F.rand(20) * 3).cast("int") + 1))
    .withColumn("cidade", F.element_at(cidades, (F.rand(21) * 5).cast("int") + 1))
    .withColumn("atualizado_em", F.current_timestamp())
)

clientes.write.mode("overwrite").json(f"{BASE}/clientes/carga_inicial")
print(f"{clientes.count()} clientes gerados")